# sft35-data: muestra de puzzles NVARC para el SFT pareado Qwen3-4B-Base vs Qwen3.5-4B-Base

CPU, 0 cuota GPU. Recorre `nvarc_full/<grupo>/*.json`, toma hasta `POR_GRUPO` puzzles por grupo con
semilla fija y escribe `/kaggle/working/nvarc_muestra.json` (lista de puzzles; puzzle = lista de pares
`{input, output}`). `sft35` lo consume como `kernel_source`.

In [ ]:
import os, json, random, time
T0 = time.time()
POR_GRUPO = int(os.environ.get("POR_GRUPO", "40"))
random.seed(0)
raiz = None
for root, dirs, files in os.walk("/kaggle/input"):
    if os.path.basename(root) == "nvarc_full":
        raiz = root; break
assert raiz, "sin nvarc_full"
grupos = sorted(os.listdir(raiz))
print("grupos:", len(grupos), "en", raiz, flush=True)
puzzles, stats = [], {"grupos": len(grupos), "ficheros": 0, "pares": 0, "celdas": 0, "saltados": 0}
for gi, g in enumerate(grupos):
    d = os.path.join(raiz, g)
    try:
        fichs = sorted(f for f in os.listdir(d) if f.endswith(".json"))
    except NotADirectoryError:
        continue
    stats["ficheros"] += len(fichs)
    random.shuffle(fichs)
    for f in fichs[:POR_GRUPO]:
        try:
            p = json.load(open(os.path.join(d, f)))
        except Exception:
            stats["saltados"] += 1; continue
        if not (isinstance(p, list) and len(p) >= 2 and all("input" in x and "output" in x for x in p)):
            stats["saltados"] += 1; continue
        ok = True
        for x in p:
            for k in ("input", "output"):
                gr = x[k]
                if not (1 <= len(gr) <= 30 and 1 <= len(gr[0]) <= 30 and all(len(r) == len(gr[0]) for r in gr)):
                    ok = False
        if not ok:
            stats["saltados"] += 1; continue
        puzzles.append({"grupo": g, "fichero": f, "pares": p})
        stats["pares"] += len(p)
        stats["celdas"] += sum(len(x[k]) * len(x[k][0]) for x in p for k in ("input", "output"))
    if gi % 50 == 0:
        print("  grupo %d/%d, puzzles %d, %.0fs" % (gi, len(grupos), len(puzzles), time.time() - T0), flush=True)
random.shuffle(puzzles)
json.dump(puzzles, open("/kaggle/working/nvarc_muestra.json", "w"))
stats["puzzles"] = len(puzzles); stats["wall_s"] = round(time.time() - T0)
stats["MB"] = round(os.path.getsize("/kaggle/working/nvarc_muestra.json") / 1e6, 1)
print("RESULT_JSON:" + json.dumps(stats), flush=True)
